# 02 – Model Selection: BART vs mDeBERTa

Benchmark zero-shot classification on real eProcure titles.

**Models:**
- `facebook/bart-large-mnli` (English-only, larger)
- `MoritzLaurer/mDeBERTa-v3-base-mnli-xnli` (multilingual, smaller)

**Hypothesis:** mDeBERTa wins because it's multilingual and smaller — critical for Indian government tenders that may contain Hindi.

**Ground truth:** 5 manually annotated representative records from the smoke data.

---

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / 'src'))
from title_cleaner import extract_title

DATA_PATH = project_root / 'data' / 'raw' / 'tenders.jsonl'
print(f'Data: {DATA_PATH} (exists={DATA_PATH.exists()})')

# Candidate labels — government tender domain
CANDIDATE_LABELS = [
    "Construction",
    "Electrical Works",
    "Medical Equipment",
    "Information Technology",
    "Roads and Bridges",
    "Maintenance and Repair",
    "Consultancy Services",
    "Supply of Goods",
    "Research and Development",
    "Security Services"
]
print(f'Candidate labels: {len(CANDIDATE_LABELS)}')

## Ground Truth: 5 Representative Tenders

Since the raw data lacks `category` labels, we manually annotate 5 representative records across categories.

In [ ]:
GROUND_TRUTH = [
    {
        "tender_id": "2026_ARMHA_906356_1",
        "raw_title": "[VIII.11011/33 AR/Engr-MW/NIT/2026-27/16] [VIII.11011/33 AR/Engr-MW/NIT/2026-27/16]",
        "ground_truth": "Maintenance and Repair"
    },
    {
        "tender_id": "2026_DREV_906625_1",
        "raw_title": "[Construction of Security Guard Booth in the Income-tax Office Campus, Tinsukia] [IT_TSK_2026-27_1]",
        "ground_truth": "Construction"
    },
    {
        "tender_id": "2026_BSF_906981_1",
        "raw_title": "[Construction of toilet block near firing range at Daizer range under STC BSF Jodhpur] [01 /NIT-COMPOSITE/FTR-RAJ/2026-27]",
        "ground_truth": "Construction"
    },
    {
        "tender_id": "2026_NIOT_903950_1",
        "raw_title": "[Running Operation Maintenance and Management of NIOT Research Vessel] [NIOT/HVT/1429/2025-26]",
        "ground_truth": "Maintenance and Repair"
    },
    {
        "tender_id": "2026_FACT_905497_1",
        "raw_title": "[Assistance for Breakdown maintenance/skilled Jobs at Offsite Area (Bagging plant, Ammonia handling area, PAT area, Ammonia jetty area, Ammonia barges, Bulk godown) FACT CD.] [4024/2025-2026/E33347]",
        "ground_truth": "Maintenance and Repair"
    },
]

for rec in GROUND_TRUTH:
    rec['clean_title'] = extract_title(rec['raw_title'])[0]

print(f'Ground truth records: {len(GROUND_TRUTH)}')
print('\nClean titles:')
for rec in GROUND_TRUTH:
    print(f"  → {rec['clean_title'][:80]}... (GT: {rec['ground_truth']})")

## Load Models

In [ ]:
from transformers import pipeline

try:
    bart = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device="cpu")
    print('✅ BART loaded successfully')
except Exception as exc:
    print(f'❌ BART failed to load: {exc}')
    bart = None

try:
    mdeberta = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli", device="cpu")
    print('✅ mDeBERTa loaded successfully')
except Exception as exc:
    print(f'❌ mDeBERTa failed to load: {exc}')
    mdeberta = None

## Evaluate on 5 Ground-Truth Records

In [ ]:
def evaluate_model(model, label):
    if model is None:
        return None
    results = []
    for rec in GROUND_TRUTH:
        pred = model(rec['clean_title'], CANDIDATE_LABELS)
        top_pred = pred['labels'][0]
        score = pred['scores'][0]
        results.append({
            'tender_id': rec['tender_id'],
            'ground_truth': rec['ground_truth'],
            'predicted': top_pred,
            'score': round(score, 4),
            'correct': top_pred == rec['ground_truth']
        })
    return results

bart_results = evaluate_model(bart, 'BART')
mdeberta_results = evaluate_model(mdeberta, 'mDeBERTa')

bart_correct = sum(r['correct'] for r in bart_results) if bart_results else 0
mdeberta_correct = sum(r['correct'] for r in mdeberta_results) if mdeberta_results else 0

print(f"BART: {bart_correct}/{len(GROUND_TRUTH)} correct")
print(f"mDeBERTa: {mdeberta_correct}/{len(GROUND_TRUTH)} correct")

## Side-by-Side Comparison

In [ ]:
comparison = []
for i in range(len(GROUND_TRUTH)):
    comparison.append({
        'tender_id': GROUND_TRUTH[i]['tender_id'],
        'title': GROUND_TRUTH[i]['clean_title'][:60] + '...',
        'ground_truth': GROUND_TRUTH[i]['ground_truth'],
        'bart_pred': bart_results[i]['predicted'] if bart_results else 'N/A',
        'bart_correct': bart_results[i]['correct'] if bart_results else False,
        'mdeberta_pred': mdeberta_results[i]['predicted'] if mdeberta_results else 'N/A',
        'mdeberta_correct': mdeberta_results[i]['correct'] if mdeberta_results else False,
    })

comp_df = pd.DataFrame(comparison)
comp_df[['tender_id', 'title', 'ground_truth', 'bart_pred', 'bart_correct', 'mdeberta_pred', 'mdeberta_correct']]

## Edge Cases: Short, Long, All-Caps, Mixed-Language

Evaluate model resilience on unusual title formats.

In [ ]:
EDGE_CASES = [
    {'name': 'Short (1 word)', 'title': 'Repair'},
    {'name': 'Long (>200 chars)', 'title': ' '.join(['Supply'] * 40)},
    {'name': 'All caps', 'title': 'CONSTRUCTION OF ROAD AND BRIDGE'},
    {'name': 'Mixed Hindi', 'title': 'सड़क निर्माण कार्य NH-157'},
    {'name': 'Nested brackets', 'title': 'Supply of [DRDO] Equipment [Model-X]'},
]

edge_results = []
for case in EDGE_CASES:
    if bart:
        bart_pred = bart(case['title'], CANDIDATE_LABELS)['labels'][0]
    else:
        bart_pred = 'N/A'

    if mdeberta:
        mdb_pred = mdeberta(case['title'], CANDIDATE_LABELS)['labels'][0]
    else:
        mdb_pred = 'N/A'

    edge_results.append({
        'case': case['name'],
        'bart': bart_pred,
        'mdeberta': mdb_pred,
    })

pd.DataFrame(edge_results)

## Decision

In [ ]:
print(f"=== DECISION ===")
print(f"BART: {bart_correct}/{len(GROUND_TRUTH)} correct")
print(f"mDeBERTa: {mdeberta_correct}/{len(GROUND_TRUTH)} correct")
print()

if bart_correct > mdeberta_correct:
    print("Decision: Use BART (better accuracy on ground truth)")
elif mdeberta_correct > bart_correct:
    print("Decision: Use mDeBERTa (better accuracy on ground truth)")
elif bart_correct == mdeberta_correct and bart_correct >= 3:
    print("Decision: Use mDeBERTa (tie, but multilingual advantage)")
else:
    print(f"WARNING: Both models performed poorly (< 3/5). Data quality issue. HALT Phase 3.")

print(f"\nRationale: mDeBERTa is smaller, multilingual, and purpose-built for cross-lingual NLI. Even if performance is comparable, it handles Hindi titles that BART cannot classify.")